# Autoencoders

_Modern Deep Learning & AI_

**Squeeze the input through a tiny bottleneck, then rebuild it. What survives is the essence.**

An autoencoder is a network that tries to copy its input to its output. But there is a catch in the middle.

     The encoder squeezes the input down to a few numbers, called the code or bottleneck. The decoder expands those few numbers back to a full reconstruction.

---

This notebook is a practice scaffold for the **Autoencoders** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Autoencoder(nn.Module):
    def __init__(self, n_in=5, code=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, 4), nn.ReLU(), nn.Linear(4, code))
        self.decoder = nn.Sequential(nn.Linear(code, 4), nn.ReLU(), nn.Linear(4, n_in))
    def forward(self, x):
        z = self.encoder(x)        # tiny code (bottleneck)
        return self.decoder(z), z  # reconstruction + code

torch.manual_seed(0)
model = Autoencoder(n_in=5, code=2)
x = torch.rand(8, 5)               # 8 synthetic samples, 5 features each
x_hat, z = model(x)
loss = F.mse_loss(x_hat, x)        # reconstruction loss
loss.backward()                    # gradients flow back through both nets
print("code shape:", z.shape)      # (8, 2) -> squeezed to 2 numbers
print("recon shape:", x_hat.shape) # (8, 5)
print("recon MSE:", round(loss.item(), 4))

## Visualize the data & results

_Encode real handwritten digits to a 2-D code: do the digit classes separate, and which images reconstruct badly?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# linear autoencoder = PCA(2) on REAL handwritten digits (8x8 images)
digits = load_digits()
X = digits.data / 16.0                  # 1797 images, 64 pixels, scaled 0..1
y = digits.target
pca = PCA(n_components=2)
Z = pca.fit_transform(X)                 # 2-D bottleneck code
recon = pca.inverse_transform(Z)         # decode back to 64 pixels
mse = np.mean((X - recon) ** 2, axis=1)  # per-image reconstruction error

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for c, col in zip([0, 1, 2], ["#4ea1ff", "#7ee787", "#ffb454"]):
    idx = np.where(y == c)[0][:20]
    axes[0].scatter(Z[idx, 0], Z[idx, 1], color=col, label="digit %d" % c)
axes[0].set_xlabel("code dim 1"); axes[0].set_ylabel("code dim 2")
axes[0].set_title("digits in the 2-D code"); axes[0].legend()

order = np.argsort(mse)
pick = list(order[200:206]) + list(order[-2:])   # typical + two hardest
lab = ["d%d" % y[i] for i in order[200:206]] + ["hard d%d" % y[i] for i in order[-2:]]
col = ["#7ee787"] * 6 + ["#ff7b72"] * 2
axes[1].bar(lab, mse[pick], color=col)
axes[1].set_title("reconstruction MSE per image")
plt.show()